<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/19_neural_networks/19_0_PyTorch_Basics/19_0_practice_quiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Run the cell below to start the practice quiz.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

all_quizzes = {
    "19_0_1: Tensors": [
        {
            "question": "What are the two capabilities a PyTorch tensor has that a NumPy array does not?",
            "options": [
                "Device-agnostic computation (.to(device)) and automatic gradient tracking (requires_grad)",
                "Faster CPU arithmetic and built-in plotting",
                "Support for missing values and labeled axes",
                "Broadcasting and matrix multiplication"
            ],
            "answer": "Device-agnostic computation (.to(device)) and automatic gradient tracking (requires_grad)"
        },
        {
            "question": "PyTorch tensors default to float32 while NumPy arrays default to float64. Why did PyTorch choose float32?",
            "options": [
                "Float32 is twice as fast on GPU hardware",
                "Float32 is more numerically precise",
                "Float64 is not supported by Python",
                "Float32 is required by the autograd engine"
            ],
            "answer": "Float32 is twice as fast on GPU hardware"
        },
        {
            "question": "What is the difference between torch.from_numpy(arr) and torch.tensor(arr)?",
            "options": [
                "from_numpy shares memory with the original array; tensor() always makes a copy",
                "from_numpy makes a copy; tensor() shares memory",
                "from_numpy only works for 1-D arrays",
                "There is no difference \u2014 they are aliases"
            ],
            "answer": "from_numpy shares memory with the original array; tensor() always makes a copy"
        },
        {
            "question": "t.sum() returns a 0-dimensional tensor. How do you get a plain Python number from it?",
            "options": [
                "Call .item() on the result",
                "Call float() on the tensor's .shape",
                "Index it with [0]",
                "It is already a Python float \u2014 nothing needed"
            ],
            "answer": "Call .item() on the result"
        }
    ],
    "19_0_2: Autograd": [
        {
            "question": "After x = torch.tensor(3.0, requires_grad=True); y = x**2; y.backward(), what does x.grad hold?",
            "options": [
                "6.0 \u2014 the derivative of x\u00b2 evaluated at x = 3",
                "9.0 \u2014 the value of y",
                "2.0 \u2014 the exponent of the operation",
                "3.0 \u2014 the value of x"
            ],
            "answer": "6.0 \u2014 the derivative of x\u00b2 evaluated at x = 3"
        },
        {
            "question": "Calling .backward() three times in a row without zeroing produces x.grad values of 6, 12, 18 instead of 6, 6, 6. Why?",
            "options": [
                "Gradients accumulate into .grad by default; each backward call adds to what is already there",
                "The learning rate triples the gradient each step",
                "The computation graph grows deeper with each call",
                "PyTorch multiplies the gradient by the call count"
            ],
            "answer": "Gradients accumulate into .grad by default; each backward call adds to what is already there"
        },
        {
            "question": "In the training-step order, when is optimizer.zero_grad() called and why?",
            "options": [
                "At the start of each step, so the new backward pass writes fresh gradients instead of adding to stale ones",
                "After optimizer.step(), to free memory used by the parameters",
                "Only once, before the first epoch",
                "Immediately after loss.backward(), to undo the gradient computation"
            ],
            "answer": "At the start of each step, so the new backward pass writes fresh gradients instead of adding to stale ones"
        },
        {
            "question": "When should a forward pass be wrapped in torch.no_grad()?",
            "options": [
                "Whenever the result will never need a .backward() call \u2014 validation, metrics, inference",
                "Whenever the model contains more than one layer",
                "During training, to speed up the backward pass",
                "Never \u2014 it disables the model's weights"
            ],
            "answer": "Whenever the result will never need a .backward() call \u2014 validation, metrics, inference"
        }
    ],
    "19_0_3: Gradient Descent": [
        {
            "question": "Why do neural networks need gradient descent when linear regression does not?",
            "options": [
                "Linear regression has a closed-form analytic solution; neural networks have no such formula, so iterative search is the only option",
                "Neural networks have too few parameters for a formula to apply",
                "Gradient descent is more accurate than analytic solutions",
                "sklearn does not support neural networks"
            ],
            "answer": "Linear regression has a closed-form analytic solution; neural networks have no such formula, so iterative search is the only option"
        },
        {
            "question": "What is the gradient descent parameter update rule?",
            "options": [
                "w = w \u2212 learning_rate \u00d7 w.grad",
                "w = w + learning_rate \u00d7 w.grad",
                "w = w.grad \u2212 learning_rate",
                "w = learning_rate \u00d7 w"
            ],
            "answer": "w = w \u2212 learning_rate \u00d7 w.grad"
        },
        {
            "question": "In the learning-rate experiment, what happened with lr = 1.5 (too large)?",
            "options": [
                "The loss overshot the minimum, bounced to larger and larger values, and diverged within a few steps",
                "The loss converged but more slowly than lr = 0.1",
                "The loss reached exactly zero in one step",
                "The loss oscillated forever between two fixed values"
            ],
            "answer": "The loss overshot the minimum, bounced to larger and larger values, and diverged within a few steps"
        },
        {
            "question": "After 300 epochs, the gradient-descent line and sklearn's analytic line were indistinguishable. What is the right takeaway?",
            "options": [
                "Gradient descent converges to the same answer; for linear regression the analytic shortcut is preferable, but for networks no shortcut exists",
                "Gradient descent is always preferable to analytic solutions",
                "sklearn secretly uses gradient descent for LinearRegression",
                "The two lines only matched because the data was standardized"
            ],
            "answer": "Gradient descent converges to the same answer; for linear regression the analytic shortcut is preferable, but for networks no shortcut exists"
        }
    ]
}

class QuizApp:
    def __init__(self, quiz_name, data):
        self.quiz_name = quiz_name
        self.data = data
        self.current_index = 0
        self.score = 0

        self.header = widgets.HTML(value="<h2>" + quiz_name + "</h2>")
        self.progress = widgets.HTML(value="")
        self.question_text = widgets.HTML(value="")
        self.options = widgets.RadioButtons(options=[], layout={"width": "max-content"})
        self.submit_btn = widgets.Button(description="Submit Answer", button_style="primary")
        self.next_btn = widgets.Button(description="Next Question", button_style="info", disabled=True)
        self.back_btn = widgets.Button(description="Back to Menu", button_style="warning")
        self.feedback = widgets.HTML(value="")

        self.submit_btn.on_click(self.handle_submit)
        self.next_btn.on_click(self.handle_next)
        self.back_btn.on_click(self.handle_back)
        self.render_question()

    def render_question(self):
        clear_output(wait=True)
        q = self.data[self.current_index]
        self.progress.value = "<b>Question " + str(self.current_index + 1) + " of " + str(len(self.data)) + "</b>"
        self.question_text.value = "<p style='font-size:14px;'>" + q["question"] + "</p>"
        self.options.options = q["options"]
        self.options.value = None
        self.submit_btn.disabled = False
        self.next_btn.disabled = True
        self.feedback.value = ""
        display(self.header, self.progress, self.question_text, self.options,
                widgets.HBox([self.submit_btn, self.next_btn]), self.feedback)

    def handle_submit(self, b):
        if self.options.value is None:
            self.feedback.value = "<span style='color:red;'>Please select an option!</span>"
            return
        correct = self.data[self.current_index]["answer"]
        if self.options.value == correct:
            self.score += 1
            self.feedback.value = "<span style='color:green; font-weight:bold;'>Correct!</span>"
        else:
            self.feedback.value = "<span style='color:red; font-weight:bold;'>Incorrect.</span><br>The correct answer was: <b>" + correct + "</b>"
        self.submit_btn.disabled = True
        self.next_btn.disabled = False

    def handle_next(self, b):
        self.current_index += 1
        if self.current_index < len(self.data):
            self.render_question()
        else:
            self.show_results()

    def handle_back(self, b):
        show_menu()

    def show_results(self):
        clear_output()
        pct = (self.score / len(self.data)) * 100
        html = """<div style="border: 2px solid #0366d6; padding: 20px; border-radius: 10px;">
            <h2>Quiz Complete!</h2>
            <p style="font-size:16px;">Your Final Score: <b>""" + str(self.score) + " / " + str(len(self.data)) + """</b></p>
            <p style="font-size:20px; color:#0366d6;"><b>Percentage: """ + "{:.1f}".format(pct) + "%</b></p></div>"
        display(widgets.HTML(html), self.back_btn)

def show_menu():
    clear_output(wait=True)
    header = widgets.HTML(value="<h2>19_0 PyTorch Basics — Practice Quizzes</h2>")
    instruction = widgets.HTML(value="<p>Select a notebook quiz below to begin.</p>")
    quiz_selector = widgets.Dropdown(options=list(all_quizzes.keys()), description="Select Quiz:", layout={"width": "50%"})
    start_btn = widgets.Button(description="Start Quiz", button_style="success")

    def on_start(b):
        QuizApp(quiz_selector.value, all_quizzes[quiz_selector.value])

    start_btn.on_click(on_start)
    display(header, instruction, quiz_selector, start_btn)

show_menu()